# Temporary: raw-image signature training (no preprocessing)

Trains the Stage 3 ConvNeXt-Tiny embeddings on the **raw** CEDAR + BHSig260 images
with `--resize-only`, so no Stage 2.2 crop / whitening / binarization is applied.

This notebook also saves the training log, plots the training curves, evaluates on the
held-out test signers, and sweeps a few fixed similarity thresholds.

**Before running:** upload `sigver_raw.zip` as a Kaggle Dataset (it extracts to
`manifest.csv` + `raw/...`), enable a GPU accelerator, and turn on Internet (for the
ImageNet weights).

In [ ]:
# --- Setup: code + dependencies ---------------------------------------------
%cd /kaggle/working
!rm -rf sigver && git clone <your-repo-url> sigver
%cd /kaggle/working/sigver
!pip install -q timm opencv-python-headless matplotlib

# Raw .tif scans make OpenCV print a harmless "TIFF Software tag" warning per file.
import os
os.environ["OPENCV_LOG_LEVEL"] = "ERROR"

In [ ]:
# --- Paths -------------------------------------------------------------------
DATA = "/kaggle/input/sigver-raw"        # dataset root: manifest.csv + raw/
RUN = "ckpt_raw"                          # checkpoint folder under /kaggle/working
CKPT = f"/kaggle/working/{RUN}/best_model.pt"
LOG = "/kaggle/working/training_log.txt"  # epoch JSON lines for analyze_log
CURVES = "/kaggle/working/curves.png"     # training-curve plot
EVAL_JSON = "/kaggle/working/eval_default.json"

!ls $DATA && head -2 $DATA/manifest.csv

In [ ]:
# --- Train on RAW images (no Stage 2.2 preprocessing) ------------------------
# --resize-only    : read raw scans, grayscale, resize to 192x384, normalize
# --negatives auto : EER/accuracy vs skilled forgeries (present in this dataset)
!python -m training.train_siamese --manifest $DATA/manifest.csv \
  --resize-only --negatives auto \
  --output-dir /kaggle/working/{RUN} \
  --device cuda --workers 2 \
  --phase1-epochs 5 --phase2-epochs 30 --lr-schedule cosine --select-by eer \
  2>&1 | tee {LOG}

In [ ]:
# --- Training curves ---------------------------------------------------------
!python -m training.analyze_log {LOG} --plot {CURVES}

from IPython.display import Image, display
display(Image(CURVES))

In [ ]:
# --- Evaluate on the held-out test signers -----------------------------------
# Threshold is chosen on the validation signers (EER) and applied to test.
!python -m training.evaluate --checkpoint {CKPT} --manifest $DATA/manifest.csv \
  --resize-only --negatives auto \
  --device cuda --workers 2 --bootstrap 1000 \
  2>&1 | tee {EVAL_JSON}

In [ ]:
# --- Accuracy at fixed similarity thresholds (bootstrap off for speed) -------
for t in [0.7, 0.8, 0.85, 0.9]:
    print("=== threshold", t)
    !python -m training.evaluate --checkpoint {CKPT} --manifest $DATA/manifest.csv \
      --resize-only --negatives auto \
      --device cuda --workers 2 --bootstrap 0 --threshold {t} \
      2>&1 | grep -E "threshold_used|accuracy_skilled|far_skilled_at_threshold|frr_at_threshold" | head -5

In [ ]:
# --- Saved checkpoint --------------------------------------------------------
!ls -lh {CKPT}

### Notes

- **Saving the model:** use **Save Version -> Save & Run All (Commit)** so the run
  survives closing the tab, then download `best_model.pt` from the Output tab.
- **Thresholds:** the default evaluation picks the threshold at the validation EER and
  applies it unchanged to the test signers. The sweep cell applies fixed thresholds
  (`--threshold`) for comparison; a threshold chosen by looking at test results is not
  an honest estimate.
- **Resize mode:** `--resize-mode stretch` forces the image to 192x384 instead of the
  default aspect-preserving resize with white padding.
- **Negatives:** `--negatives auto` uses skilled forgeries when present (this dataset
  has them) and falls back to random/different-signer pairs otherwise.